# **Saving and Loading Model Checkpoints**

We have added general-purpose methods to our `BaseGNNModel` that allow saving and loading the full model **along with its fitted encoder and architecture metadata**.

**Why is this important?**

* You no longer need to manually track architecture parameters (like hidden size, layers, etc.)
* You preserve the label encoder alongside the model weights
* You can reload the entire model and use it immediately for inference or continued training


## Load molecules, train model and perform inference.

Before introducing the save and load methods, we will repeat the process of building our model and performing inference from the "Tutorial_quickstart.ipynb"

In [ ]:
from torch.utils.data import random_split
import numpy as np
import logging

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("dataset_loader.log"),
        logging.StreamHandler()
    ]
)


from atoMLtype.datasets.GNNdataset import GNNdataset
from atoMLtype.models.ModelEncoder import ModelEncoder
from atoMLtype.analysis.accuracy_counts import plot_atom_distribution
from atoMLtype.models.GNN.DMPNNmodel import AtomBondMPNN
from atoMLtype.models.ModelTrainer import GNNTrainer
from atoMLtype.models.ModelEngine import ModelEngine


In [ ]:
dataset_encoder = ModelEncoder(collapse=True)

In [ ]:
# Load the ZINC files
zinc_sdf_path = "./data/parm_at_Frosst/zinc.sdf"
zinc_json_labels = "./data/antechamber/atomLabels_gaff2.json"

zinc_dataset = GNNdataset(sdf_path=zinc_sdf_path, 
                          label_path=zinc_json_labels, 
                          directed_graph=True,
                          labeled=True,
                          encoder=dataset_encoder)

In [ ]:
# Split Train and test dataset (90% train, 10% test)
train_size = int(0.90 * len(zinc_dataset))
test_size = len(zinc_dataset) - train_size
train_dataset, test_dataset = random_split(zinc_dataset, [train_size, test_size])

In [ ]:

atom_node_dim = train_dataset[0].x.shape[1]
bond_edge_dim = train_dataset[0].edge_attr.shape[1]

AtomMPNN_zinc = AtomBondMPNN(atom_input_dim=atom_node_dim, 
                             bond_input_dim=bond_edge_dim, 
                             hidden_dim=512,
                             encoder=dataset_encoder, 
                             num_layers=10,
                             use_attention=True)

In [ ]:
trainer = GNNTrainer(AtomMPNN_zinc, 
                     dataset=train_dataset, 
                     batch_size=32, learning_rate=0.001,
                     epochs=5, 
                     k_folds=5, 
                     random_seed=21)

In [ ]:
training_loss_ouput = trainer.train(draw_curve=False,
                                    verbose=False,
                                    report_step=5
                                    )

In [ ]:
modelEngine_zinc = ModelEngine(model=AtomMPNN_zinc, 
                          dataset=test_dataset, 
                          device="cpu",
                          batch_size=32)

In [ ]:
pred_record = modelEngine_zinc.predict(analysis=True)

pred_record.summary()

## **Saving a Model**

After training, simply call:

In [ ]:
AtomMPNN_zinc.save(filepath="./saved_models/tutorial_saveModel.mdl")

This will store:

* The model weights (`state_dict`)
* The fitted `ModelEncoder`
* The architecture metadata from the model’s `get_metadata()` method

## **Loading a Model**

To reload the saved model:


In [ ]:
import atoMLtype.models.BaseGNNModel
import atoMLtype.models.GNN.DMPNNmodel
import importlib
importlib.reload(atoMLtype.models.BaseGNNModel)
importlib.reload(atoMLtype.models.GNN.DMPNNmodel)
from atoMLtype.models.GNN.DMPNNmodel import AtomBondMPNN

loaded_model = AtomBondMPNN.load("./saved_models/tutorial_saveModel.mdl")

This:

* Reconstructs the model architecture using the saved metadata
* Restores the weights
* Restores the encoder (so label mappings are preserved)


## Exmaple usage and results with the loaded model

Every works with the model as before

In [ ]:
loaded_modelEngine_zinc = ModelEngine(model=loaded_model, 
                          dataset=test_dataset, 
                          device="cpu",
                          batch_size=32)

pred_record_loaded_model = loaded_modelEngine_zinc.predict(analysis=True)

pred_record_loaded_model.summary()

The summary output is exact the same as before with the loaded model. 

### **NOTES**

1. It is important to get the encoder from the loaded model when using with new GNNDataset. With the loaded and prefitted encoder the GNNDataset will filter the molecules that are labelled to only work with this prefitted encoder. 

2. **The `_get_metadata()` Abstract Method**

Each `BaseGNNModel` subclass must implement:

```python
def get_metadata(self) -> dict:
    """
    Returns architecture metadata needed to reconstruct the model.

    Example:
        return {
            'atom_input_dim': self.atom_input_dim,
            'bond_input_dim': self.bond_input_dim,
            'num_layers': self.num_layers,
            'hidden_dim': self.hidden_dim,
            'use_attention': self.use_attention
        }
    """
```

- This ensures the **correct configuration** is saved and later used by `.load()`
- Without this, you would have to supply all architecture parameters manually when loading.



## **NOTES**

### 1. **Important: Use the Encoder from the Loaded Model**

When you load a model checkpoint and want to apply it to a new `GNNdataset`,
make sure you **use the encoder attached to the loaded model**:



This ensures the `GNNdataset` only processes molecules with labels **that match** the prefitted encoder.
Otherwise, you risk label mismatches or missing label mappings.




In [ ]:
# Load the model and get its attached encoder
loaded_encoder = loaded_model.encoder

zinc_sdf_path = "./data/parm_at_Frosst/zinc.sdf"
zinc_json_labels = "./data/antechamber/atomLabels_gaff2.json"

# Pass this encoder to the new dataset
new_dataset = GNNdataset(
    sdf_path=zinc_sdf_path,
    label_path=zinc_json_labels,
    directed_graph=True,
    labeled=True,
    encoder=loaded_encoder
)


### 2. **The `_get_metadata()` Abstract Method**

Each `BaseGNNModel` subclass must implement a `get_metadata()` method
to return **all necessary architecture parameters** for reconstructing the model during load.

Example:

```python
def get_metadata(self) -> dict:
    """
    Returns architecture metadata needed to reconstruct the model.

    Example:
        return {
            'atom_input_dim': self.atom_input_dim,
            'bond_input_dim': self.bond_input_dim,
            'num_layers': self.num_layers,
            'hidden_dim': self.hidden_dim,
            'use_attention': self.use_attention
        }
    """
```

Why this matters:

* Guarantees the saved checkpoint can **fully self-describe** its architecture.
* Eliminates the need to supply `load()` with manual parameters.
* Makes the save/load system robust and easy to use across pipelines.